# new_table

Makes a Table folder for each new Saved table: from its query, when QUERIES gives one, its logic written for you; else from a Template, its logic for you to write. A Building block, a piece several notebooks share, is made with project.new_block(...).

## Terms

- **Step**: each numbered heading below, with the code cell under it. Its first line says its kind: it changes only files, or only prints; it is a call to the warehouse, yours to run; or it is a decision, yours to make.
- **Names cell**: the code cell after the one that finds the Project. It holds this Workflow's inputs, the start materials you fill in once, each with what to write in its comment; each step's call reads them.
- **Decisions cell**: `DECISIONS`, the code cell after the names cell, where you answer each open Decision a step waits on, by the table's name.
- **Table folder**: the folder in tables/ that holds everything about one Saved table: its Table notebook, its past lineage, and its Hive.
- **Saved table**: a table in the warehouse that only your own code writes, its files in a folder of its own.
- **Template**: the empty shape of a Table notebook, in admin/templates/, each specific left as a `<...>` to fill in.
- **Building block**: a piece of your own code several notebooks share, a file of its own in building_blocks/.
- **Decision**: one thing about one table that only you decide, such as its location or its Key; the table's files say whether it is answered yet.
- **Table reference**: a script in references/ describing one table: its columns and their types, its Partitions and its Key.
- **Batch**: the scripts converted together, under one name.
- **Table notebook**: the notebook in a Table folder whose cells create, preview and write its Saved table; its logic cell is the one whose first line is `# logic`.
- **Key**: the columns whose values name exactly one row of a table.
- **Dev Database**: the database where you create and write tables of your own, each built, written and compared there first: `DEV` in admin/databases.py.
- **Hive**: the SQL your warehouse runs.
- **Partition**: the rows of a table that share one value of each column it is partitioned by.

## Before you start

Fill in the names cell below and save the notebook: each call reads it, and so does `project.check()`, which names the next step.

## Decision rules

- If you want the steps run for you, as most do: fill in the names cell and save the notebook, run "Run up to your Decisions", paste what it prints into the decisions cell under it, fill it in and save, then run "Run the rest". Each runs every step left in turn, up to the first only you can do, and says where it stopped.
- If you run the steps yourself, run each step's cell under "Step by step", in turn, in order.
- If you want only the steps that change files run for you, run `project.do_next()`, the last cell. It runs them up to the first it can't.

## Self-check

Once the last step is done, each of these holds:

- Each table in `TABLES` has its Table folder in tables/, and its step 2 cell runs with no error.
- Each table's Table reference lists its columns and leaves no Decision open.
- admin/sent.csv records each table created and written.
- Each table made from a query has a `conversion/compare.md` that says it passed.
- `project.check()` names no next step of this Workflow.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import connect, query, execute

# Your connection to the warehouse. Each notebook runs on its own, so each connects here, at
# its top, before it calls query or execute: write yours in the line below and delete its #,
# such as connect(your_library.connect(...)). A connection kept in a file outside the Project,
# such as team_connection.py, is imported by these lines, written in that line's place:
#     sys.path.append("C:/work/connections")  # the folder team_connection.py is in
#     from team_connection import conn
#     connect(conn)
from sqlglot_composer import example_database
sys.path.append('C:\\Users\\myfir\\AppData\\Local\\Temp\\usability-loop\\runs\\2026-11-09-W8-2')  # the folder team_connection.py is in
from team_connection import conn
connect(conn)

In [ ]:
TABLES = []  # the new tables, by their names in the warehouse, such as ["mart_dev.daily_job_runs"]
QUERIES = {}  # each table's query, by its name without its database, as Hive text or a .hql file in admin/workflows/new_table/, such as {"daily_job_runs": "daily_job_runs.hql"} for mart_dev.daily_job_runs: it is made in your Dev Database, its logic written from its query

## Run up to your Decisions

This cell and "Run the rest" are one call, `project.run_workflow`, run once before you answer and once after. Run the cell below: it runs every step left, up to the first only you can do, and says where it stopped. At a Decision, it prints a whole DECISIONS: paste it into the decisions cell under it, fill in each <FILL IN>, and save the notebook. With nothing waiting on you, it runs every step to the end, its calls to the warehouse too: give it dry_run=True first to see what it would send.

In [ ]:
# Run up to your Decisions: every step left, up to the first only you can do, such as a
# Decision, whose DECISIONS it prints to paste into the cell under it.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("new_table", query=query, execute=execute)

In [ ]:
DECISIONS = {}  # your answer to each open Decision, by the table's name as
# project.check() writes it, such as
# {"mart_dev.weekly_ops": {"key": ["job_id"], "types": {"fails": "bigint"},
#                          "tags": {"PERIOD_END": "'2026-10-31'"}}}
# "key": None says no columns name one row. project.check() names each one open, and
# prints a whole DECISIONS to paste over this line: fill in each <FILL IN>. A Key
# convert or new_table proposed, marked # proposed, is kept unless you answer "key"
# here; one add_references proposed stays open until you accept it.
# "key": "proposed" accepts either as if typed. "all_tables" answers for every table
# of this notebook, and a table's own answer overwrites it, kind by kind and tag by
# tag:
# {"all_tables": {"key": "proposed",
#                 "tags": {"PERIOD_END": "'2026-10-31'"}}}
# accepts each table's proposed Key and gives each table holding PERIOD_END that
# value. A table's location is placed under "location_root" in
# admin/databases.py; answer "location" only for one it can't place.
# A tag may be answered with a Run value's function of your Project's, imported above
# DECISIONS, such as "PERIOD_END": period_end. For a table's latest Partition, write
# project.new_run_value("period_end", latest_partition="ops.job_runs.dt") at the top
# of this cell, the import it prints under it, and run the cell once: it writes the
# function into building_blocks/run_values.py. And "seed", such as
# {"dt": {"from": "2026-01-01"}}, limits the Partitions a table that reads itself
# copies: how-to 09 shows both.

## Run the rest

Run the cell below once the decisions cell above is filled in and saved: the same call again, it goes on from where the cell before stopped, and it too stops at the first step only you can do: a refusal, a compare that failed, or a Decision left open, each named. Mend what it names, such as an answer in the decisions cell, save, and run this cell again.

In [ ]:
# Run the rest: the same call, run again once you have answered: it goes
# on from where it stopped, and stops again at a refusal, a compare that
# failed or a Decision left open.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("new_table", query=query, execute=execute)

# Step by step

Each step's own cell, to run or run again one step on purpose; the cells above run them all for you.

## 1. Make each Table folder, converting each table's query QUERIES gives

It changes only files, so project.do_next() runs it for you.

- **Goal:** a Table folder for each table in `TABLES`: from its query, when `QUERIES` gives one, its logic written for you; else from a Template, its logic for you to write.
- **Action:** If `QUERIES` gives a query, first give each table it reads a Table reference, with the references Workflow. Then run the cell below. Each query is converted as the convert Workflow converts one: in a batch of its own, named for its table, in `admin/workflows/convert/`; the batch moves to `finished/` once compared.
- **Done when** each table in `TABLES` has its Table folder in tables/; then step 2.

In [ ]:
project.new_table(TABLES, queries=QUERIES)

## 2. Write each table's logic in its Table notebook, and save it

It is a decision, yours to make.

- **Goal:** logic in each Table notebook that loads, so its `preview()` builds.
- **Action:** For each table in `TABLES`, open its Table notebook, in its Table folder. If `QUERIES` gives the table's query, its logic is written already, from that query: read its logic cell, the one whose first line is `# logic`, and each `# CONVERSION:` line in it, which `project.check()` lists too, and change nothing unless you want to, since step 8 compares its rows with the query's. If `QUERIES` gives it no query, fill in each `<...>` in its code cells as its text cell says, and save it. Then run the cell below: it says what stops each table's logic, if anything does.
- **Done when** the cell below runs with no error; then step 3.

In [ ]:
for table in TABLES:
    project.load(table).preview()  # says what stops it, if anything does

## 3. Write each Table reference from its logic

It changes only files, so project.do_next() runs it for you.

- **Goal:** each table's Table reference written from its logic: its columns and types from `preview()`, its partition columns from `PARTITIONED_BY`, and the Key its logic makes.
- **Action:** Run the cell below. It names each TODO line each Table reference leaves, such as its docstring's first line, which waits on nothing. If it says it kept a Table reference, as it does for a table made from its query, that file is yours already, its columns written. Either way, open each Table reference it names, fill in each TODO line it names by hand, such as one line saying what one row is, and save the file, now or when you like: no step waits on them.
- **Done when** each table's Table reference in references/ lists its columns, no longer the stand-in new_table wrote; then step 4.

In [ ]:
project.reference_from_logic(TABLES)

## 4. Answer each table's open Decisions in the decisions cell

It is a decision, yours to make.

- **Goal:** an answer to each open Decision of the new tables.
- **Action:** Run the cell below, `project.check()`: it names each open Decision and prints a whole `DECISIONS` to paste over the decisions cell's; then write each answer in the decisions cell, `DECISIONS`, by the table's name as `project.check()` writes it, then save the notebook.
- **Done when** the decisions cell answers each Decision `project.check()` names for these tables; then step 5.

In [ ]:
project.check()

## 5. Write each answer into its table's Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** each answer written into its table's Table reference.
- **Action:** Run the cell below.
- **Done when** each table's Table reference leaves no Decision open; then step 6.

In [ ]:
project.write_decisions(DECISIONS, tables=TABLES)

## 6. Check each table's rows with preview(), which writes nothing

It is a call to the warehouse, yours to run.

- **Goal:** each table's rows read and checked before anything is written.
- **Action:** Run the cell below. It prints the rows each table's `preview()` gives, read through your query. `project.check()` names this step beside step 7.
- **Done when** you have read each table's rows, and they are the rows it should hold; then step 7.

In [ ]:
for table in TABLES:
    print(run_query(project.load(table).preview(), query))

## 7. Create and write each in dev

It is a call to the warehouse, yours to run.

- **Goal:** each table made and written in your Dev Database.
- **Action:** Run the cell below. It sends each table's `create()`, then `project.run_tables` sends each table's `write()`, through your execute.
- **Done when** admin/sent.csv records each table written; then step 8.

In [ ]:
for table in TABLES:
    run_execute(project.load(table).create(), execute)
project.run_tables(tables=TABLES, execute=execute)

## 8. Compare each table made from a query with its query

It is a call to the warehouse, yours to run.

- **Goal:** each table made from a query checked against its query.
- **Action:** Run the cell below. `project.compare_batch` compares each with its query, through your query, and writes the result in its `conversion/compare.md`. If `QUERIES` gives no query, this step is done once step 7 is.
- **Done when** each table made from a query has a `conversion/compare.md` that says it passed; the Workflow is done: go through the Self-check at the top.

In [ ]:
for name in QUERIES:
    print(project.compare_batch(name, query=query))

In [ ]:
project.do_next()